[ADBC](https://arrow.apache.org/adbc/) (Arrow Database Connectivity) is a database access API designed around [Apache Arrow](https://arrow.apache.org/), enabling efficient columnar data interchange between applications and databases. Unlike traditional row-oriented interfaces, ADBC allows you to retrieve query results directly as Arrow tables, avoiding costly row-to-column conversions.

This notebook walks through connecting to [MySQL](https://www.mysql.com/) from Python using ADBC with the standard DBAPI interface. You'll learn how to execute queries, fetch results as Arrow tables, perform bulk ingestion of Arrow data, and inspect database metadata.

Requirements:

- Python 3
- MySQL or Docker

## Setup

Install the required dependencies:

In [3]:
%pip install -q dbc adbc-driver-manager pyarrow

Note: you may need to restart the kernel to use updated packages.


Install the MySQL ADBC driver:

In [ ]:
!dbc install -q mysql

If you don't already have a MySQL instance running, start an instance with Docker:

In [6]:
!docker run -d --rm --name some-mysql -e MYSQL_ROOT_PASSWORD=my-secret-pw -p 3306:3306 mysql

cc50bb23986d58770ff684f9cce2db7435826968d51cd57ded17378e3de40d09


Import PyArrow and the ADBC driver manager:

In [7]:
import pyarrow as pa
from adbc_driver_manager import dbapi

## Connection and Cursor

Create a DBAPI-style connection with the MySQL database:

In [9]:
connection = dbapi.connect(
    driver="mysql",
    db_kwargs={"uri": "root:my-secret-pw@tcp(localhost:3306)/mysql"},
    autocommit=True,
)

Create a cursor:

In [10]:
cursor = connection.cursor()

## Query Execution

Execute a SQL query and get the results via the normal, row-oriented DBAPI interface:

In [11]:
cursor.execute("SELECT 1 AS id, 'Alice' AS name")
cursor.fetchone()

(1, 'Alice')

Alternatively, get the results as a PyArrow Table:

In [12]:
cursor.execute("SELECT 1 AS id, 'Alice' AS name")
cursor.fetch_arrow_table()

pyarrow.Table
id: int64 not null
name: string not null
----
id: [[1]]
name: [["Alice"]]

Parameters can be bound to queries:

In [13]:
cursor.execute("SELECT ? + 1 AS favorite_num", parameters=(10,))
cursor.fetch_arrow_table()

pyarrow.Table
favorite_num: int64
----
favorite_num: [[11]]

## Bulk Ingestion

Ingest Arrow data into a database table:

In [14]:
table = pa.table({"id": [1, 2, 3, 4], "name": ["Ian", "Matt", "David", "Bryce"]})
cursor.adbc_ingest(table_name="users", data=table, mode="create")

4

Append to the database table:

In [15]:
table = pa.table({"id": [5, 6], "name": ["Mandy", "Sam"]})
cursor.adbc_ingest(table_name="users", data=table, mode="append")

2

Query the ingested data:

In [16]:
cursor.execute("SELECT * FROM users")
cursor.fetchall()

[(1, 'Ian'), (2, 'Matt'), (3, 'David'), (4, 'Bryce'), (5, 'Mandy'), (6, 'Sam')]

## Metadata

Get information about the database and driver:

In [17]:
connection.adbc_get_info()

{'vendor_name': 'MySQL',
 'vendor_version': '9.6.0 (MySQL Community Server - GPL)',
 'vendor_arrow_version': '(unknown or development build)',
 3: True,
 4: False,
 'driver_name': 'ADBC Driver Foundry Driver for MySQL',
 'driver_version': 'v0.3.1',
 'driver_arrow_version': 'v18.5.2',
 'driver_adbc_version': 1001000}

Query for tables and columns in the database:

In [18]:
info = (
    connection.adbc_get_objects(catalog_filter="mysql", table_name_filter="users")
    .read_all()
    .to_pylist()
)
catalog = info[0]
schema = catalog["catalog_db_schemas"][0]
tables = schema["db_schema_tables"]

In [19]:
tables[0]["table_name"]

'users'

In [20]:
[column["column_name"] for column in tables[0]["table_columns"]]

['id', 'name']

Get the Arrow schema of a table:

In [21]:
connection.adbc_get_table_schema("users")

id: int64
  -- field metadata --
  sql.column_name: 'id'
  sql.database_type_name: 'bigint'
  sql.precision: '19'
  sql.scale: '0'
name: string
  -- field metadata --
  sql.column_name: 'name'
  sql.database_type_name: 'text'
  sql.length: '65535'

## Cleanup

Close the cursor and connection:

In [22]:
cursor.close()
connection.close()

If you ran MySQL with Docker earlier, stop and remove the container:

In [23]:
!docker stop some-mysql

some-mysql
